In [2]:
from google.colab import files

uploaded = files.upload()

Saving seasonal_model_df_encoded.csv to seasonal_model_df_encoded.csv


In [3]:
import pandas as pd

lstm_df = pd.read_csv(
    "seasonal_model_df_encoded.csv"
)

print(lstm_df.shape)

(4196, 41)


In [4]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import MinMaxScaler

In [5]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from tensorflow.keras.callbacks import EarlyStopping

In [6]:
print(lstm_df.columns.tolist())

['period', 'date', 'year', 'month', 'quarter', 'flowDesc', 'primaryValue', 'crude_oil_avg_usd_lag1', 'energy_index_lag1', 'non_energy_index_lag1', 'food_index_lag1', 'fertilizers_index_lag1', 'base_metals_index_lag1', 'precious_metals_index_lag1', 'fed_funds_rate_lag1', 'usd_broad_index_lag1', 'us_industrial_production_lag1', 'us_cpi_lag1', 'us_10y_treasury_rate_lag1', 'global_policy_uncertainty_lag1', 'deep_sea_freight_ppi_lag1', 'log_primaryValue', 'trade_value_lag12', 'country_name_Australia', 'country_name_Brazil', 'country_name_Canada', 'country_name_China', 'country_name_France', 'country_name_Germany', 'country_name_India', 'country_name_Indonesia', 'country_name_Italy', 'country_name_Japan', 'country_name_Mexico', 'country_name_Republic_of_Korea', 'country_name_Saudi_Arabia', 'country_name_South_Africa', 'country_name_Turkiye', 'country_name_United_Kingdom', 'country_name_United_States', 'flowCode_X']


In [7]:
lstm_df["date"] = pd.to_datetime(
    lstm_df["date"]
)

lstm_df = lstm_df.sort_values(
    ["date"]
)

In [8]:
country_cols = [
    col for col in lstm_df.columns
    if col.startswith("country_name_")
]

In [9]:
print(len(country_cols))
print(country_cols[:5])

17
['country_name_Australia', 'country_name_Brazil', 'country_name_Canada', 'country_name_China', 'country_name_France']


In [10]:
lstm_df["Country"] = "Argentina"

for col in country_cols:
    country = col.replace("country_name_", "")
    lstm_df.loc[lstm_df[col] == 1, "Country"] = country

In [11]:
print(lstm_df["Country"].value_counts())

Country
Argentina            238
Republic_of_Korea    238
Mexico               238
Japan                238
Australia            238
Italy                238
India                238
Germany              238
France               238
Canada               238
Turkiye              238
Brazil               238
United_Kingdom       238
United_States        238
Indonesia            234
South_Africa         234
Saudi_Arabia         204
China                192
Name: count, dtype: int64


In [12]:
lstm_df["Flow"] = "Import"

lstm_df.loc[
    lstm_df["flowCode_X"] == 1,
    "Flow"
] = "Export"

In [13]:
print(lstm_df["Flow"].value_counts())

Flow
Import    2098
Export    2098
Name: count, dtype: int64


In [14]:
lstm_df = lstm_df.sort_values(
    ["Country", "Flow", "date"]
)

lstm_df = lstm_df.reset_index(drop=True)

In [15]:
feature_cols = [
    "log_primaryValue",
    "trade_value_lag12",
    "month",
    "quarter"
]

In [16]:
external_lag_cols = [
    col for col in lstm_df.columns
    if col.endswith("_lag1")
]

In [17]:
feature_cols = feature_cols + external_lag_cols

print("Number of features:", len(feature_cols))

Number of features: 18


In [18]:
X_sequences = []
y_sequences = []
sequence_dates = []
sequence_countries = []
sequence_flows = []

In [19]:
sequence_length = 12

In [20]:
feature_cols = (
    feature_cols
    + country_cols
    + ["flowCode_X"]
)

print("Total features:", len(feature_cols))

Total features: 36


In [21]:
for (country, flow), group in lstm_df.groupby(["Country", "Flow"]):
    group = group.sort_values("date").reset_index(drop=True)
    for i in range(sequence_length, len(group)):
        X_sequences.append(group[feature_cols].iloc[i-12:i].values)
        y_sequences.append(group["log_primaryValue"].iloc[i])
        sequence_dates.append(group["date"].iloc[i])
        sequence_countries.append(country); sequence_flows.append(flow)

In [22]:
X_sequences = np.array(X_sequences)
y_sequences = np.array(y_sequences)

sequence_dates = np.array(sequence_dates)

In [23]:
print(X_sequences.shape)
print(y_sequences.shape)

(3764, 12, 36)
(3764,)


In [26]:
train_mask = sequence_dates < np.datetime64("2024-01-01")
test_mask = sequence_dates >= np.datetime64("2024-01-01")

TypeError: Cannot compare Timestamp with datetime.date. Use ts == pd.Timestamp(date) or ts.date() == date instead.

In [27]:
sequence_dates = pd.to_datetime(sequence_dates)

train_mask = sequence_dates < pd.Timestamp("2024-01-01")
test_mask = sequence_dates >= pd.Timestamp("2024-01-01")

In [28]:
X_train_lstm = X_sequences[train_mask]
X_test_lstm = X_sequences[test_mask]

y_train_lstm = y_sequences[train_mask]
y_test_lstm = y_sequences[test_mask]

In [29]:
print("Training:", X_train_lstm.shape)
print("Test:", X_test_lstm.shape)
print("Training target:", y_train_lstm.shape)
print("Test target:", y_test_lstm.shape)

Training: (3332, 12, 36)
Test: (432, 12, 36)
Training target: (3332,)
Test target: (432,)


In [30]:
input_scaler = MinMaxScaler()

X_train_2d = X_train_lstm.reshape(
    -1, X_train_lstm.shape[2]
)

In [31]:
input_scaler.fit(X_train_2d)

X_train_scaled_2d = input_scaler.transform(
    X_train_2d
)

In [32]:
X_test_2d = X_test_lstm.reshape(
    -1, X_test_lstm.shape[2]
)

X_test_scaled_2d = input_scaler.transform(
    X_test_2d
)

In [33]:
X_train_scaled = X_train_scaled_2d.reshape(
    X_train_lstm.shape
)

X_test_scaled = X_test_scaled_2d.reshape(
    X_test_lstm.shape
)

In [34]:
print(X_train_scaled.shape)
print(X_test_scaled.shape)

(3332, 12, 36)
(432, 12, 36)


In [35]:
target_scaler = MinMaxScaler()

y_train_scaled = target_scaler.fit_transform(
    y_train_lstm.reshape(-1, 1)
)

In [36]:
lstm_model = Sequential()

lstm_model.add(
    LSTM(50, input_shape=(12, 36))
)

lstm_model.add(Dense(1))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [37]:
lstm_model.compile(
    optimizer="adam",
    loss="mse"
)

In [38]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

In [39]:
history = lstm_model.fit(
    X_train_scaled,
    y_train_scaled,
    epochs=100,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping],
    verbose=1
)

Epoch 1/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0167 - val_loss: 0.0466
Epoch 2/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 0.0014 - val_loss: 0.0638
Epoch 3/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 8.3939e-04 - val_loss: 0.0648
Epoch 4/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 7.0077e-04 - val_loss: 0.0607
Epoch 5/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 6.0879e-04 - val_loss: 0.0586
Epoch 6/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 5.7328e-04 - val_loss: 0.0596
Epoch 7/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - loss: 5.5444e-04 - val_loss: 0.0552
Epoch 8/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 9ms/step - loss: 5.3422e-04 - val_loss: 0.0575
Epoch 9/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 5.0696e-04 - val_loss: 0.0504
Epoch 10/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 4.7693e-04 - val_loss: 0.0514
Epoch 11/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 4.7206e-04 - val_loss: 0.0526


In [40]:
lstm_predictions_scaled = lstm_model.predict(
    X_test_scaled
)

14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 30ms/step


In [41]:
lstm_predictions_log = target_scaler.inverse_transform(
    lstm_predictions_scaled
).flatten()

In [42]:
lstm_predictions = np.expm1(
    lstm_predictions_log
)

lstm_actual = np.expm1(
    y_test_lstm
)

In [44]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

In [45]:
lstm_mae = mean_absolute_error(
    lstm_actual, lstm_predictions
)

lstm_rmse = np.sqrt(
    mean_squared_error(lstm_actual, lstm_predictions)
)

In [46]:
lstm_r2 = r2_score(
    lstm_actual, lstm_predictions
)

lstm_mape = np.mean(
    np.abs((lstm_actual - lstm_predictions) / lstm_actual)
) * 100

In [47]:
print("LSTM Results")
print("------------")
print("MAE:", round(lstm_mae / 1e9, 2), "billion USD")
print("RMSE:", round(lstm_rmse / 1e9, 2), "billion USD")
print("R2:", round(lstm_r2, 4))
print("MAPE:", round(lstm_mape, 2), "%")

LSTM Results
------------
MAE: 19.85 billion USD
RMSE: 42.69 billion USD
R2: 0.6303
MAPE: 27.34 %


In [48]:
lstm_results = pd.DataFrame({
    "Model": ["LSTM"],
    "MAE (USD bn)": [lstm_mae / 1_000_000_000],
    "RMSE (USD bn)": [lstm_rmse / 1_000_000_000],
    "R2": [lstm_r2],
    "MAPE (%)": [lstm_mape]
})

In [49]:
lstm_results.round(3)

,Model,MAE (USD bn),RMSE (USD bn),R2,MAPE (%)
0,LSTM,19.852,42.689,0.63,27.343
